In [ ]:
#!pip install langgraph
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
import re


# Carrega as variáveis de ambiente e configura a chave da API da OpenAI
import os
from dotenv import load_dotenv

from groq import Groq

from langchain_groq import ChatGroq

# Inicializa o modelo Groq
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key="gsk_"
)



In [2]:
class State(TypedDict):
    """ Representa o estado do processo de avaliação da redação """
    essay: str
    relevance_score: float
    grammar_score: float
    structure_score: float
    depth_score: float
    final_score: float

In [ ]:
%pip install -U langchain-groq


In [9]:
import re

def extract_score(text: str) -> float:
    match = re.search(
        r"(?:Pontuação|Score|Nota)\s*:\s*\**\s*([0-9]+(?:[.,][0-9]+)?)",
        text,
        re.IGNORECASE
    )

    if not match:
        raise ValueError(
            f"Não foi possível extrair a pontuação de: {text}"
        )

    score = float(match.group(1).replace(",", "."))

    if not 0.0 <= score <= 1.0:
        raise ValueError(f"Pontuação fora do intervalo [0, 1]: {score}")

    return score
def check_relevance(state: State) -> State:
    """Verifica a relevância da redação."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a relevância da seguinte redação em relação ao tema dado, prezando pela excelência da Língua Portuguesa. "
        "Forneça uma pontuação de relevância entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["relevance_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_relevance: {e}")
        state["relevance_score"] = 0.0
    return state

def check_grammar(state: State) -> State:
    """Verifica a gramática da redação."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a gramática da Língua Portuguesa na seguinte redação. "
        "Forneça uma pontuação de gramática entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["grammar_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_grammar: {e}")
        state["grammar_score"] = 0.0
    return state

def analyze_structure(state: State) -> State:
    """Analisa a estrutura da redação."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a estrutura de acordo com a normal culta da Língua Portuguesa na seguinte redação. "
        "Forneça uma pontuação de estrutura entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["structure_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em analyze_structure: {e}")
        state["structure_score"] = 0.0
    return state

def evaluate_depth(state: State) -> State:
    """Avalia a profundidade de análise na redação."""
    prompt = ChatPromptTemplate.from_template(
        "Avalie a profundidade de análise na seguinte redação. "
        "Forneça uma pontuação de profundidade entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nRedação: {essay}"
    )
    result = llm.invoke(prompt.format(essay=state["essay"]))
    try:
        state["depth_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em evaluate_depth: {e}")
        state["depth_score"] = 0.0
    return state

def calculate_final_score(state: State) -> State:
    """Calcula a pontuação final com base nas pontuações dos componentes individuais."""
    state["final_score"] = (
        state["relevance_score"] * 0.3 +
        state["grammar_score"] * 0.2 +
        state["structure_score"] * 0.2 +
        state["depth_score"] * 0.3
    )
    return state
    

In [10]:
# Inicializa o StateGraph
workflow = StateGraph(State)

# Adiciona nós ao grafo
workflow.add_node("check_relevance", check_relevance)
workflow.add_node("check_grammar", check_grammar)
workflow.add_node("analyze_structure", analyze_structure)
workflow.add_node("evaluate_depth", evaluate_depth)
workflow.add_node("calculate_final_score", calculate_final_score)

# Define e adiciona arestas condicionais
workflow.add_conditional_edges(
    "check_relevance",
    lambda x: "check_grammar" if x["relevance_score"] > 0.5 else "calculate_final_score"
)
workflow.add_conditional_edges(
    "check_grammar",
    lambda x: "analyze_structure" if x["grammar_score"] > 0.6 else "calculate_final_score"
)
workflow.add_conditional_edges(
    "analyze_structure",
    lambda x: "evaluate_depth" if x["structure_score"] > 0.7 else "calculate_final_score"
)
workflow.add_conditional_edges(
    "evaluate_depth",
    lambda x: "calculate_final_score"
)

# Define o ponto de entrada
workflow.set_entry_point("check_relevance")

# Define o ponto de saída
workflow.add_edge("calculate_final_score", END)


# Compila o grafo
app = workflow.compile()

In [11]:
def grade_essay(essay: str) -> dict:
    """Avalia a redação fornecida usando o fluxo de trabalho definido."""
    initial_state = State(
        essay=essay,
        relevance_score=0.0,
        grammar_score=0.0,
        structure_score=0.0,
        depth_score=0.0,
        final_score=0.0
    )
    result = app.invoke(initial_state)
    return result

In [12]:
sample_essay = """
# Claude Code: Arquitetura Agêntica, Funcionalidades Avançadas e Transformação dos Processos de Engenharia de Software

A evolução das tecnologias de inteligência artificial aplicadas à engenharia de software tem provocado mudanças significativas na concepção, implementação, manutenção e gerenciamento de sistemas computacionais. Nesse cenário, o **Claude Code**, desenvolvido pela Anthropic, representa uma abordagem avançada de assistência à programação fundamentada na utilização de modelos de linguagem de grande escala, mecanismos de raciocínio contextual e execução autônoma de ferramentas computacionais. Diferentemente dos assistentes convencionais, cuja principal função consiste em sugerir fragmentos de código ou fornecer respostas pontuais a problemas de programação, o Claude Code foi concebido para atuar diretamente sobre ambientes reais de desenvolvimento, permitindo interpretar estruturas complexas de projetos, modificar arquivos, executar comandos, investigar falhas, implementar funcionalidades e participar de processos completos de desenvolvimento de software. Sua relevância decorre não apenas da capacidade de produzir código sintaticamente válido, mas principalmente da possibilidade de integrar atividades de análise, planejamento, execução e verificação em um fluxo operacional relativamente autônomo, embora sujeito a mecanismos de autorização, supervisão humana e restrições de segurança.

Uma das características fundamentais do Claude Code é sua **arquitetura agêntica**, baseada na interação entre um modelo de inteligência artificial e um conjunto de ferramentas que possibilitam observar e modificar o ambiente computacional. Essa arquitetura pode ser compreendida como um ciclo iterativo de percepção, interpretação, planejamento, ação e avaliação. Inicialmente, o sistema recebe uma solicitação expressa em linguagem natural, interpreta os objetivos propostos e identifica quais informações são necessárias para realizar a tarefa. Em seguida, utiliza ferramentas de leitura, pesquisa, navegação por diretórios e inspeção de arquivos para construir uma representação contextual do projeto. A partir dessas informações, estabelece uma sequência de operações que pode envolver edição de código, instalação de dependências, execução de scripts ou aplicação de testes automatizados. Os resultados obtidos retornam ao modelo como novas evidências, permitindo revisar hipóteses, identificar problemas e executar ações adicionais. Essa capacidade de adaptação distingue um agente de programação de um sistema de geração textual isolado, pois transforma a interação em um processo computacional iterativo, no qual decisões subsequentes dependem dos efeitos efetivamente observados.

No contexto da compreensão de bases de código, o Claude Code apresenta funcionalidades particularmente relevantes para projetos de grande porte, sistemas legados e arquiteturas compostas por numerosos módulos interdependentes. O agente pode explorar a estrutura hierárquica dos diretórios, identificar arquivos de configuração, localizar definições de funções, examinar interfaces, interpretar dependências e investigar os caminhos pelos quais determinadas informações são processadas. Em aplicações organizadas segundo princípios de modularidade, orientação a objetos ou arquitetura de microsserviços, essa capacidade de investigação permite reconstruir parcialmente relações entre componentes e identificar regiões do sistema potencialmente afetadas por uma modificação. Por exemplo, diante da necessidade de alterar um mecanismo de autenticação, o agente pode examinar simultaneamente as rotas responsáveis pelo recebimento das credenciais, os serviços que executam sua validação, os modelos utilizados para persistência de dados e os testes associados ao comportamento esperado. Embora a qualidade dessa análise dependa do contexto disponível e da complexidade do repositório, a integração entre pesquisa semântica, inspeção textual e execução de ferramentas reduz a necessidade de localizar manualmente cada componente relacionado ao problema.

Outra funcionalidade central consiste na **implementação contextualizada de funcionalidades de software**, na qual o agente deixa de atuar exclusivamente sobre instruções isoladas e passa a considerar as convenções arquiteturais existentes. Mediante uma descrição funcional, o Claude Code pode propor e implementar novas classes, funções, módulos, interfaces e rotinas de processamento. Essa atividade pode incluir a criação de arquivos, alteração de componentes existentes, atualização de configurações e adaptação de testes. Sua operação não está necessariamente restrita a uma única linguagem de programação, uma vez que utiliza ferramentas compatíveis com o ambiente disponível e interpreta diferentes linguagens, formatos de configuração e estruturas de projeto. Em sistemas que combinam Python, TypeScript, Rust, SQL e outras tecnologias, por exemplo, o agente pode examinar relações entre diferentes camadas e sugerir modificações coordenadas. Entretanto, a consistência das alterações depende da qualidade das especificações, das convenções fornecidas e dos mecanismos de verificação utilizados, não sendo possível presumir que todo código gerado seja funcionalmente correto apenas por apresentar uma estrutura aparentemente coerente.

A **depuração automatizada** constitui outra capacidade de destaque. Em vez de limitar-se à interpretação abstrata de mensagens de erro, o Claude Code pode investigar falhas utilizando informações produzidas pela execução real de comandos e programas. O processo envolve examinar rastreamentos de exceções, interpretar mensagens de compiladores, consultar arquivos de log, localizar instruções potencialmente responsáveis por comportamentos inesperados e formular hipóteses sobre a origem das falhas. Essas hipóteses podem orientar modificações específicas, seguidas da execução de testes destinados a verificar se o problema foi efetivamente corrigido. Tal abordagem aproxima o agente de um processo experimental de diagnóstico, no qual cada tentativa produz evidências que podem corroborar ou refutar uma explicação inicial. Em aplicações distribuídas, por exemplo, uma falha aparentemente associada a uma requisição HTTP pode decorrer de erros de serialização, contratos inconsistentes entre serviços ou configurações incorretas de ambiente. O Claude Code pode auxiliar nessa investigação, desde que tenha acesso aos componentes, registros e ferramentas relevantes. Apesar disso, falhas intermitentes, dependentes de hardware ou associadas a condições de concorrência continuam exigindo validação rigorosa.

No domínio da qualidade de software, o Claude Code também pode atuar na **geração, execução e manutenção de testes automatizados**. Sua atuação abrange testes unitários, testes de integração, verificações de regressão e, conforme a infraestrutura disponível, testes funcionais de aplicações. O agente pode identificar comportamentos esperados, elaborar casos de teste, construir entradas representativas e verificar se as saídas produzidas correspondem às especificações estabelecidas. Além disso, consegue executar ferramentas externas como sistemas de teste, analisadores estáticos, formatadores e compiladores. Essa integração permite estruturar ciclos nos quais uma alteração de código é imediatamente submetida a procedimentos de validação. Em ambientes científicos e de engenharia, a mesma abordagem pode ser aplicada à implementação de experimentos numéricos, à criação de testes de propriedades matemáticas e à comparação de resultados computacionais com referências conhecidas. Deve-se reconhecer, contudo, que uma suíte de testes bem-sucedida demonstra apenas conformidade com as propriedades efetivamente verificadas; ela não constitui uma prova universal da ausência de defeitos, tampouco substitui análises formais quando estas são necessárias.

A capacidade de **refatoração estrutural** amplia consideravelmente o campo de aplicação do Claude Code. Por meio da análise de módulos existentes, o agente pode identificar duplicações, dependências desnecessárias, funções excessivamente extensas, inconsistências de nomenclatura e oportunidades de reorganização. A partir dessa avaliação, pode propor a decomposição de componentes, a extração de interfaces, a centralização de responsabilidades ou a substituição de implementações pouco legíveis por alternativas mais organizadas. Em sistemas complexos, uma refatoração pode exigir modificações simultâneas em numerosos arquivos, preservando contratos e comportamentos externos. O agente pode coordenar parte dessas alterações e verificar seus efeitos por meio da infraestrutura de testes. Todavia, a preservação da equivalência funcional deve ser tratada como uma propriedade a demonstrar, não como consequência automática da refatoração. Em sistemas críticos, especialmente aqueles submetidos a restrições temporais, regulatórias ou de segurança, alterações arquiteturais precisam considerar também desempenho, previsibilidade, consumo de recursos e impactos sobre componentes externos.

Um diferencial relevante é a integração direta com o **terminal e o sistema operacional**, que permite transformar instruções expressas em linguagem natural em procedimentos executáveis. Quando autorizado, o Claude Code pode executar comandos para pesquisar arquivos, consultar o estado de um repositório, instalar dependências, compilar aplicações, iniciar processos e executar scripts. Essa funcionalidade reduz a distância entre o raciocínio sobre um problema e sua resolução operacional, pois o agente não precisa limitar-se a sugerir comandos para execução posterior pelo usuário. Em vez disso, pode executar uma sequência controlada de operações e interpretar seus resultados. A consequência prática é a possibilidade de automatizar fluxos que envolvem várias etapas interdependentes, como preparar um ambiente de desenvolvimento, reproduzir uma falha, corrigir uma implementação, atualizar testes e verificar novamente a aplicação. Entretanto, as ações possíveis dependem das permissões concedidas, do sistema operacional, das ferramentas instaladas e das restrições aplicadas ao ambiente.

A interação com sistemas de **controle de versão**, especialmente Git, representa outra dimensão fundamental. O Claude Code pode auxiliar na interpretação de alterações entre versões, investigação de históricos de desenvolvimento, identificação de regressões, preparação de commits e organização de modificações destinadas a pull requests. Sua integração com plataformas de hospedagem de código permite também participar de processos de revisão, desde que as credenciais e permissões necessárias estejam configuradas. Em ambientes colaborativos, essa capacidade pode contribuir para a análise de mudanças propostas, identificação de inconsistências e geração de correções relacionadas a comentários de revisão. O suporte a GitHub Actions amplia essas possibilidades ao permitir a execução de tarefas agênticas em fluxos de integração contínua. Desse modo, determinadas solicitações registradas em issues ou pull requests podem originar processos automatizados de análise e implementação. Ainda assim, a incorporação de um agente em pipelines de desenvolvimento exige atenção às permissões do repositório, à procedência dos comandos executados e aos impactos de alterações realizadas automaticamente.

No que se refere à organização do trabalho, o Claude Code oferece mecanismos de **planejamento antes da execução**, particularmente importantes em tarefas complexas ou potencialmente destrutivas. Em um modo de planejamento, o agente pode investigar o repositório, compreender restrições e propor uma estratégia de implementação sem modificar imediatamente o código-fonte. Essa separação entre análise e execução introduz uma oportunidade de inspeção humana antes de mudanças significativas. Em projetos que envolvem migrações de banco de dados, reorganizações arquiteturais ou substituição de dependências fundamentais, esse procedimento permite examinar antecipadamente os arquivos envolvidos, os riscos previstos, as condições de sucesso e os possíveis mecanismos de reversão. A estratégia aproxima o desenvolvimento assistido por inteligência artificial de processos estruturados de engenharia, nos quais decisões de implementação são precedidas por levantamento de requisitos, análise de consequências e definição de critérios de aceitação. A autonomia operacional, nesse contexto, não elimina a importância de especificações explícitas; ao contrário, torna essas especificações ainda mais relevantes.

A **persistência de contexto e instruções de projeto** constitui outro elemento essencial da plataforma. Por meio de arquivos como `CLAUDE.md`, e de mecanismos adicionais de memória e regras de configuração, o desenvolvedor pode estabelecer convenções que orientam o comportamento do agente em diferentes sessões. Essas instruções podem especificar padrões de codificação, arquitetura desejada, comandos de teste, restrições de dependências, requisitos de documentação e procedimentos de validação. Em um projeto de software embarcado, por exemplo, seria possível registrar a necessidade de evitar alocação dinâmica, restringir determinadas bibliotecas, preservar compatibilidade com uma arquitetura de hardware ou respeitar requisitos de execução em ambientes sem sistema operacional. O agente pode utilizar essas informações como contexto persistente, reduzindo a repetição de instruções em tarefas posteriores. Entretanto, persistência de instruções não equivale a garantia absoluta de cumprimento, sendo recomendável complementar convenções textuais com verificações automatizadas capazes de detectar violações objetivas.

A extensibilidade do Claude Code é reforçada pelo mecanismo de **skills**, que permite definir procedimentos reutilizáveis contendo instruções especializadas e, quando apropriado, materiais auxiliares. Uma skill pode representar um fluxo de auditoria de código, um procedimento de documentação, uma rotina de análise de segurança ou uma metodologia de implementação específica de determinado domínio. Essas definições podem ser acionadas explicitamente ou selecionadas pelo agente conforme a relevância para a tarefa. A principal vantagem está na possibilidade de transformar conhecimento operacional recorrente em uma capacidade reutilizável, reduzindo a dependência de prompts extensos e repetitivos. Em ambientes organizacionais, skills compartilhadas podem contribuir para padronizar tarefas como revisão de código, preparação de versões, atualização de documentação e execução de verificações de qualidade. Sua eficácia, entretanto, depende da precisão das instruções, da adequação das ferramentas disponíveis e da manutenção contínua desses procedimentos diante da evolução do projeto.

Complementarmente, os **hooks** possibilitam integrar mecanismos programáticos a eventos específicos do ciclo de execução do agente. Diferentemente de uma orientação escrita em linguagem natural, um hook pode acionar comandos ou verificações associados a eventos determinados, introduzindo um nível adicional de automatização e controle. É possível utilizar esses mecanismos, conforme sua configuração, para verificar condições antes da execução de ferramentas, executar procedimentos após determinadas alterações ou registrar eventos relevantes de uma sessão. Em um ambiente de engenharia de software, tal recurso pode auxiliar na aplicação de políticas de qualidade, na execução de formatadores, na verificação de determinadas restrições e na integração com sistemas externos de observabilidade. A distinção fundamental é que instruções textuais dependem da interpretação do modelo, enquanto hooks corretamente configurados podem executar operações determinísticas diante de eventos definidos. Essa combinação permite construir ambientes em que decisões flexíveis orientadas por inteligência artificial coexistem com mecanismos tradicionais de automação verificável.

Entre os recursos de integração, o **Model Context Protocol (MCP)** possui importância estratégica por oferecer um mecanismo padronizado de conexão entre o agente e ferramentas ou fontes de dados externas. Por meio de servidores MCP compatíveis, o Claude Code pode acessar capacidades que não fazem parte de seu conjunto básico de ferramentas, incluindo serviços de desenvolvimento, plataformas colaborativas, bancos de dados e sistemas corporativos. Isso permite ampliar sua atuação para além da manipulação direta de arquivos locais. Em um fluxo de trabalho integrado, por exemplo, o agente pode consultar uma especificação armazenada em uma ferramenta externa, relacioná-la com a implementação existente, realizar modificações autorizadas e produzir informações úteis para acompanhamento da tarefa. O benefício central do MCP está na extensibilidade da interface entre raciocínio e ação. Todavia, cada integração acrescenta uma superfície de acesso que deve ser administrada cuidadosamente, especialmente quando envolve credenciais, informações confidenciais ou operações capazes de modificar recursos externos.

A utilização de **subagentes especializados** introduz uma dimensão adicional de modularidade. Em vez de concentrar toda a investigação, interpretação e execução em uma única sequência de contexto, o Claude Code pode delegar determinadas atividades a agentes auxiliares. Cada subagente pode operar com instruções próprias, um conjunto específico de ferramentas e uma janela de contexto independente. Essa organização é particularmente útil quando uma tarefa secundária exige a leitura de numerosos arquivos, análise extensa de registros ou produção de informações que não precisam permanecer integralmente no contexto principal. Um subagente pode, por exemplo, investigar dependências de determinado módulo, enquanto outro examina possibilidades de teste ou realiza uma revisão especializada de código. Os resultados relevantes retornam ao fluxo principal, permitindo que o agente coordenador utilize conclusões sem incorporar necessariamente todos os dados intermediários. Essa arquitetura favorece a decomposição de problemas e a especialização funcional, embora também introduza desafios de coordenação, consumo adicional de recursos e eventual inconsistência entre análises independentes.

Em configurações que oferecem execução paralela, equipes de agentes ou sessões independentes, torna-se possível distribuir atividades de desenvolvimento entre diferentes unidades de trabalho. Essa distribuição pode representar uma vantagem em projetos nos quais determinadas tarefas apresentam baixa dependência mútua, como análise de módulos distintos, preparação de testes e investigação documental. Entretanto, o paralelismo não deve ser confundido com aceleração garantida. Quando dois agentes modificam componentes compartilhados, podem surgir conflitos de integração, inconsistências arquiteturais e duplicações de esforço. Consequentemente, a eficiência de uma abordagem multiagente depende da qualidade da decomposição, da definição das interfaces entre tarefas e da existência de mecanismos de integração e validação. Sob uma perspectiva de engenharia, o problema aproxima-se da coordenação de processos concorrentes: aumentar o número de executores pode elevar a capacidade de processamento, mas também ampliar custos de sincronização e riscos de interferência.

Outra transformação significativa decorre da disponibilidade do Claude Code em **diferentes superfícies de execução**, incluindo terminal, extensões de ambientes integrados de desenvolvimento, aplicações desktop e sessões em nuvem. No terminal, a interação privilegia comandos, arquivos e ferramentas existentes no sistema operacional. Em ambientes como Visual Studio Code e IDEs compatíveis, o agente pode trabalhar próximo ao contexto de edição do desenvolvedor, facilitando a análise de trechos selecionados e a inspeção de alterações. Nas interfaces desktop, recursos visuais podem auxiliar na revisão de diferenças entre arquivos, no acompanhamento de sessões e na verificação de aplicações. Já as sessões remotas permitem executar tarefas em ambientes computacionais separados da máquina local, conforme os recursos oferecidos e as permissões disponíveis. Essa diversidade de interfaces contribui para adaptar a ferramenta a diferentes estratégias de desenvolvimento, embora determinadas funcionalidades dependam da plataforma, do plano contratado e da configuração específica utilizada.

A execução remota também possibilita novas formas de organização do ciclo de desenvolvimento. Atividades como migração de dependências, execução de conjuntos extensos de testes, implementação de funcionalidades e revisão de grandes repositórios podem ser encaminhadas a sessões de trabalho na nuvem. Quando disponíveis, mecanismos de isolamento por ambiente ou por estruturas de controle de versão ajudam a reduzir interferências entre atividades simultâneas. O desenvolvedor pode posteriormente examinar os resultados, revisar diferenças e integrar alterações aprovadas. Esse modelo modifica a relação temporal entre solicitação e execução de tarefas, permitindo que determinadas operações prossigam em um ambiente computacional independente. Contudo, é necessário distinguir isolamento operacional de garantia de correção: a existência de um ambiente separado protege parcialmente o processo de desenvolvimento, mas não assegura que as mudanças produzidas estejam livres de falhas, inadequações arquiteturais ou riscos de segurança.

No desenvolvimento de interfaces e aplicações interativas, algumas configurações do Claude Code permitem integrar **automação de navegador, inspeção visual e interação com aplicações gráficas**. Com ferramentas apropriadas, o agente pode abrir uma aplicação, examinar elementos da interface, observar mensagens de erro, interagir com formulários e utilizar evidências visuais para orientar correções. Essa capacidade é relevante porque diversas falhas de software não são facilmente identificadas apenas por análise estática: problemas de responsividade, sobreposição de elementos, estados incorretos de componentes e comportamentos inesperados durante a navegação exigem observação do sistema em funcionamento. A integração entre implementação e verificação visual permite construir ciclos de desenvolvimento mais completos. Entretanto, a disponibilidade desses mecanismos varia conforme o sistema operacional e as integrações habilitadas, e sua utilização requer atenção especial quando o navegador ou a aplicação possui acesso a dados autenticados ou operações sensíveis.

A possibilidade de utilizar o Claude Code em **pipelines de integração e entrega contínuas** amplia sua função de assistente individual para a de componente operacional em processos de engenharia. Por meio de integrações e interfaces programáticas, determinadas atividades podem ser iniciadas por eventos do repositório, comandos de automação ou solicitações estruturadas. Nesse contexto, o agente pode investigar falhas de testes, auxiliar na elaboração de correções, analisar alterações e produzir relatórios técnicos. O Claude Agent SDK estende essa arquitetura ao permitir a construção de aplicações agênticas personalizadas em linguagens como Python e TypeScript, reutilizando mecanismos de execução de ferramentas, gerenciamento de sessões, permissões e controle contextual. Essa funcionalidade é particularmente relevante para organizações interessadas em desenvolver agentes especializados que interajam com infraestruturas próprias. Entretanto, o uso em produção exige práticas tradicionais de engenharia, incluindo autenticação, controle de acesso, observabilidade, tratamento de falhas, limites operacionais e avaliação sistemática dos resultados.

Sob a perspectiva da **segurança computacional**, o Claude Code incorpora mecanismos destinados a controlar quais recursos podem ser consultados ou modificados. Entre eles estão modos de permissão, regras de autorização para ferramentas e possibilidades de isolamento de comandos. Em configurações restritivas, determinadas operações de escrita, execução de comandos ou acesso a serviços externos dependem de autorização explícita. Em outras configurações, parte dessas operações pode ser previamente autorizada conforme políticas estabelecidas. O isolamento por sandbox, quando habilitado e suportado, pode impor restrições adicionais ao sistema de arquivos e à conectividade de rede. Essas funcionalidades são especialmente importantes porque um agente com capacidade de executar comandos possui consequências operacionais muito diferentes de um sistema que apenas produz sugestões textuais. O acesso indevido a credenciais, a execução de comandos destrutivos ou a modificação inadvertida de arquivos críticos representam riscos reais que precisam ser mitigados por princípios de privilégio mínimo, segregação de ambientes e validação das ações realizadas.

A segurança deve também considerar o problema de **injeção de instruções por conteúdos não confiáveis**, particularmente relevante em sistemas que interpretam automaticamente arquivos, páginas externas, logs ou informações obtidas de ferramentas. Um agente pode encontrar conteúdos produzidos por terceiros que contenham instruções destinadas a modificar seu comportamento ou induzir ações não autorizadas. Por esse motivo, arquivos de um repositório, resultados de buscas e respostas de serviços externos devem ser tratados de acordo com sua procedência e nível de confiança. A utilização de políticas restritivas, revisão de comandos sensíveis, separação de credenciais e execução em ambientes isolados contribui para reduzir a exposição. Ainda assim, mecanismos de segurança não devem ser interpretados como garantias absolutas, principalmente em fluxos que concedem ampla autonomia operacional.

Do ponto de vista da **gestão do contexto**, a qualidade das respostas do Claude Code está diretamente relacionada às informações disponíveis durante a execução. Modelos de linguagem operam dentro de limites de contexto, e grandes quantidades de código, registros e resultados de ferramentas podem competir por espaço com instruções e objetivos relevantes. Para administrar esse problema, a plataforma dispõe de mecanismos de gerenciamento de sessões, compactação contextual e delegação de tarefas. A seleção adequada das informações fornecidas ao agente constitui, portanto, uma decisão de engenharia. Instruções excessivamente genéricas podem produzir implementações incompatíveis com requisitos específicos, enquanto contextos volumosos e pouco organizados podem comprometer a identificação das restrições fundamentais. Uma utilização tecnicamente consistente exige definir objetivos verificáveis, fornecer especificações pertinentes, orientar o acesso às fontes relevantes e estabelecer critérios claros para avaliar os resultados.

Essa discussão conduz a uma questão central: a **diferença entre autonomia operacional e confiabilidade técnica**. O fato de um agente conseguir executar diversas etapas de desenvolvimento não implica que suas decisões sejam necessariamente corretas, ótimas ou compatíveis com todos os requisitos não explicitados. O Claude Code pode produzir hipóteses inadequadas, interpretar incorretamente relações entre componentes, modificar arquivos desnecessários ou propor soluções que funcionam em cenários simples, mas falham diante de condições excepcionais. Em aplicações que exigem determinismo, garantias temporais, segurança funcional ou preservação de invariantes matemáticos, essas limitações assumem importância ainda maior. Por exemplo, em sistemas de controle, uma implementação numericamente plausível não demonstra estabilidade, robustez ou respeito às restrições físicas do processo. Da mesma forma, uma otimização computacional aparentemente eficiente pode alterar propriedades de precisão numérica, concorrência ou consumo de memória. Nessas situações, a inteligência artificial deve apoiar a investigação e a implementação, enquanto as garantias formais e experimentais continuam dependendo de métodos apropriados de validação.

A utilização acadêmica e científica do Claude Code também merece destaque. Em projetos de pesquisa computacional, o agente pode auxiliar na organização de experimentos, implementação de algoritmos, criação de estruturas de teste e documentação de procedimentos. Em aplicações de inteligência artificial e aprendizado de máquina, por exemplo, pode contribuir para a estruturação de pipelines de treinamento, definição de rotinas de avaliação, investigação de incompatibilidades entre bibliotecas e automatização de verificações de reprodutibilidade. Em projetos que envolvem sistemas dinâmicos, otimização matemática ou computação científica, pode auxiliar na implementação de métodos numéricos, elaboração de experimentos comparativos e verificação computacional de propriedades específicas. Seu uso, entretanto, não substitui a necessidade de fundamentação matemática, análise de hipóteses e interpretação científica dos resultados. Um código executável pode implementar um modelo incorreto, e resultados numericamente consistentes podem refletir premissas inadequadas.

Também se destaca a dimensão **econômica e operacional** da ferramenta. A execução de tarefas agênticas pode envolver numerosas chamadas ao modelo, consultas a ferramentas, processamento de grandes quantidades de informações e execução repetida de verificações. Consequentemente, a complexidade das tarefas influencia custos, consumo de recursos e tempo de execução. A delegação a subagentes, o processamento paralelo e a utilização de contextos extensos podem ampliar a capacidade operacional, mas também elevar o consumo total. A avaliação da eficiência do Claude Code deve considerar, portanto, não apenas o tempo necessário para produzir uma primeira implementação, mas o esforço de revisão, a quantidade de correções posteriores, a qualidade do resultado e a manutenção futura. Em organizações, o ganho real de produtividade depende da relação entre trabalho automatizado, supervisão necessária e custo de erros eventualmente introduzidos.

Em termos organizacionais, a adoção do Claude Code pode modificar a distribuição das responsabilidades no processo de desenvolvimento. Atividades tradicionalmente executadas de maneira manual, como pesquisa no repositório, preparação de alterações repetitivas, execução de testes e elaboração inicial de documentação, tornam-se candidatas à automação supervisionada. Com isso, parte do esforço humano pode ser redirecionada para arquitetura, definição de requisitos, avaliação de riscos, estabelecimento de invariantes e análise de qualidade. Entretanto, essa transformação exige competências adicionais: o profissional precisa compreender como formular solicitações verificáveis, inspecionar alterações produzidas por agentes, identificar decisões tecnicamente inadequadas e construir mecanismos que limitem comportamentos indesejados. A capacidade de utilizar ferramentas agênticas de forma produtiva depende, paradoxalmente, de conhecimento sólido sobre os fundamentos da engenharia de software.

Em uma interpretação mais ampla, o Claude Code representa a convergência entre **modelos de linguagem, automação computacional, engenharia de contexto e sistemas de execução orientados por ferramentas**. Seu funcionamento pode ser associado a uma arquitetura de controle de tarefas, na qual um componente decisório interpreta objetivos, consulta o estado disponível, seleciona ações e utiliza resultados observados para orientar decisões posteriores. Essa analogia, embora útil conceitualmente, não implica equivalência com sistemas de controle formalmente especificados, uma vez que as decisões do modelo são probabilísticas e dependentes do contexto. Ainda assim, a estrutura evidencia uma mudança fundamental: a inteligência artificial deixa de operar apenas como mecanismo de recomendação e passa a participar diretamente da execução de procedimentos computacionais, sob restrições definidas pela infraestrutura e pelos operadores humanos.

    """

In [13]:
# Avalia a redação de exemplo
result = grade_essay(sample_essay)

# Converte as pontuações de 0-1 para 0-10
final_score = result['final_score'] * 10
relevance_score = result['relevance_score'] * 10
grammar_score = result['grammar_score'] * 10
structure_score = result['structure_score'] * 10
depth_score = result['depth_score'] * 10

# Exibe os resultados
print(f"Pontuação Final da Redação: {final_score:.2f}/10\n")
print(f"Pontuação de Relevância: {relevance_score:.2f}/10")
print(f"Pontuação de Gramática: {grammar_score:.2f}/10")
print(f"Pontuação de Estrutura: {structure_score:.2f}/10")
print(f"Pontuação de Profundidade: {depth_score:.2f}/10")

Pontuação Final da Redação: 8.80/10

Pontuação de Relevância: 9.60/10
Pontuação de Gramática: 9.30/10
Pontuação de Estrutura: 8.60/10
Pontuação de Profundidade: 7.80/10


In [17]:
try:
    result = grade_essay(sample_essay)
except Exception as e:
    print("Tipo:", type(e).__name__)
    print("Detalhes:", str(e))

Tipo: OpenAIRateLimitError
Detalhes: Error code: 429 - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.
